In [1]:
import numpy as np
import pandas as pd

In [2]:
train_path = "../data/processed/train_clean.csv"
test_path = "../data/processed/test_clean.csv"

In [4]:
train_fe = pd.read_csv("../data/processed/train_clean.csv")
test_fe = pd.read_csv("../data/processed/test_clean.csv")


In [7]:
print(train_fe.shape)
print(test_fe.shape)

print(train_fe.dtypes.value_counts())
print(test_fe.dtypes.value_counts())

(635, 5752)
(403, 5751)
float64    5750
str           2
Name: count, dtype: int64
float64    5750
str           1
Name: count, dtype: int64


In [9]:
feature_cols = train_fe.columns.drop(['id', 'target'])

In [10]:
parts = feature_cols.to_series().str.extract(r'^(t\d+)_(.+)$')

time_counts = parts.groupby(1)[0].nunique()

print("Channels:", time_counts.size)
print("Minimum time steps:", time_counts.min())
print("Maximum time steps:", time_counts.max())
print("All channels have 250 steps:", (time_counts == 250).all())

Channels: 23
Minimum time steps: 250
Maximum time steps: 250
All channels have 250 steps: True


In [12]:
X_train_imp = np.load("../data/processed/X_train_imputed.npy")
X_val_imp = np.load("../data/processed/X_val_imputed.npy")
X_test_imp = np.load("../data/processed/X_test_imputed.npy")

In [13]:
print(X_train_imp.shape)
print(X_val_imp.shape)
print(X_test_imp.shape)

(508, 5750)
(127, 5750)
(403, 5750)


In [14]:
channel = "ankle_acc_x"

cols = [f"t{i:03d}_{channel}" for i in range(250)]

indices = [list(feature_cols).index(c) for c in cols]

sample = X_train_imp[0, indices]

print(sample[:10])
print("Total readings:", len(sample))

[ 1.4364      7.6616001   7.6911998  48.88586045  7.4059      7.5984001
  7.3713002   7.5458002   7.5363002   7.5172    ]
Total readings: 250


In [15]:
print("Mean:", sample.mean())
print("STD:", sample.std())
print("Min:", sample.min())
print("Max:", sample.max())

Mean: 6.740835985346082
STD: 9.333499069746196
Min: -43.86653599358152
Max: 49.0917267450164


In [16]:
import numpy as np
import pandas as pd

train_fe = pd.read_csv("../data/processed/train_imputed.csv")
test_fe = pd.read_csv("../data/processed/test_imputed.csv")

feature_cols = train_fe.columns.drop(['id', 'target'])

X_train = train_fe[feature_cols].values
y_train = train_fe['target']

X_test = test_fe[feature_cols].values

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Channels:", len(set(
    col.split("_", 1)[1] for col in feature_cols
)))

Train: (635, 5750)
Test: (403, 5750)
Channels: 23


In [18]:
def extract_features(X, feature_cols):

    channels = sorted(set(
        col.split("_", 1)[1] for col in feature_cols
    ))

    extracted = []
    feature_names = []

    for channel in channels:

        indices = [
            i for i, col in enumerate(feature_cols)
            if col.endswith("_" + channel)
        ]

        indices = sorted(
            indices,
            key=lambda i: int(feature_cols[i].split("_")[0][1:])
        )

        signal = X[:, indices]

        mean = np.mean(signal, axis=1)
        std = np.std(signal, axis=1)
        minimum = np.min(signal, axis=1)
        maximum = np.max(signal, axis=1)
        rms = np.sqrt(np.mean(signal ** 2, axis=1))
        energy = np.sum(signal ** 2, axis=1)

        extracted.extend([
            mean, std, minimum, maximum, rms, energy
        ])

        feature_names.extend([
            f"{channel}_mean",
            f"{channel}_std",
            f"{channel}_min",
            f"{channel}_max",
            f"{channel}_rms",
            f"{channel}_energy"
        ])

    return pd.DataFrame(
        np.column_stack(extracted),
        columns=feature_names
    )

In [19]:
train_features = extract_features(X_train, feature_cols)
test_features = extract_features(X_test, feature_cols)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

train_features.head()

Train features: (635, 138)
Test features: (403, 138)


,ankle_acc_x_mean,ankle_acc_x_std,ankle_acc_x_min,ankle_acc_x_max,ankle_acc_x_rms,ankle_acc_x_energy,ankle_acc_y_mean,ankle_acc_y_std,ankle_acc_y_min,ankle_acc_y_max,...,chest_ecg_1_min,chest_ecg_1_max,chest_ecg_1_rms,chest_ecg_1_energy,chest_ecg_2_mean,chest_ecg_2_std,chest_ecg_2_min,chest_ecg_2_max,chest_ecg_2_rms,chest_ecg_2_energy
0,1.964338,8.605426,-42.131253,48.737804,8.826776,19477.994893,-8.830976,12.177687,-63.985497,42.799318,...,-10.025291,10.565194,2.128449,1132.573468,0.172441,1.726241,-8.789890,9.304897,1.734833,752.411236
1,1.233410,10.243139,-42.943454,44.847747,10.317132,26610.801164,-8.563451,12.465881,-61.879417,48.492419,...,-9.042372,9.180535,1.932418,933.559384,0.007008,1.816902,-8.620242,9.538980,1.816915,825.295361
2,0.712798,7.805598,-44.670000,45.421562,7.838076,15358.859183,-10.684434,11.484461,-63.416564,42.295809,...,-9.518518,9.715995,2.419423,1463.402054,-0.094896,1.791649,-9.516733,8.794899,1.794160,804.752685
3,4.704599,10.394657,-44.746862,45.749726,11.409739,32545.535147,-12.803829,11.693993,-63.191917,40.008120,...,-9.536631,10.009737,1.966188,966.474198,0.037959,1.673659,-9.790791,9.072521,1.674089,700.643880
4,2.863983,7.139458,-39.684267,45.739991,7.692480,14793.563276,-8.109600,12.790250,-61.218223,45.228541,...,-10.025291,8.598919,1.652873,682.996984,0.599789,1.620765,-8.883102,9.846180,1.728186,746.656535


In [20]:
print("Missing values:", train_features.isna().sum().sum())
print("Infinite values:", np.isinf(train_features).sum().sum())
print("Duplicate columns:", train_features.T.duplicated().sum())

print("Train shape:", train_features.shape)
print("Test shape:", test_features.shape)

Missing values: 0
Infinite values: 0
Duplicate columns: 0
Train shape: (635, 138)
Test shape: (403, 138)


In [21]:
train_feature_final = pd.concat(
    [
        train_fe[['id', 'target']].reset_index(drop=True),
        train_features.reset_index(drop=True)
    ],
    axis=1
)

test_feature_final = pd.concat(
    [
        test_fe[['id']].reset_index(drop=True),
        test_features.reset_index(drop=True)
    ],
    axis=1
)

train_feature_final.to_csv(
    "../data/processed/train_features_138.csv", index=False
)

test_feature_final.to_csv(
    "../data/processed/test_features_138.csv", index=False
)

print(train_feature_final.shape)
print(test_feature_final.shape)

(635, 140)
(403, 139)


In [22]:
from sklearn.model_selection import train_test_split

X = train_feature_final.drop(columns=['id', 'target'])
y = train_feature_final['target']

X_train_138, X_val_138, y_train_138, y_val_138 = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train_138.shape)
print("Validation:", X_val_138.shape)

Train: (508, 138)
Validation: (127, 138)


In [23]:
# Reuse the exact row indices from the 138-feature split
train_idx = X_train_138.index
val_idx = X_val_138.index

# Original 5750 features
X_train_5750 = train_fe.loc[train_idx, feature_cols]
X_val_5750 = train_fe.loc[val_idx, feature_cols]

# Same target labels
y_train_5750 = train_fe.loc[train_idx, 'target']
y_val_5750 = train_fe.loc[val_idx, 'target']

print("Original train:", X_train_5750.shape)
print("Original validation:", X_val_5750.shape)

print("Same training labels:",
      (y_train_5750 == y_train_138).all())

print("Same validation labels:",
      (y_val_5750 == y_val_138).all())

Original train: (508, 5750)
Original validation: (127, 5750)
Same training labels: True
Same validation labels: True
